# OddsJam EV Filter Research — Notebook 02: Filter Evaluation and Exchange Diagnostics

## Purpose

Evaluate historical executed-bet performance across saved filters and venue regimes using canonical evaluation tables, exchange liquidity-capacity diagnostics, and scorecards. These are descriptive research outputs, not validated filter recommendations.

**Input:** processed Parquet snapshot produced by `scripts/preprocess_bet_tracker.py`. Run from `notebooks/` with the project Poetry kernel. Set `AS_OF_DATE` to an existing processed snapshot suffix.

**Interpretation:** historical filter labels may not capture configuration changes; repeated executions and overlapping opportunities are not independent bets; modeled EV is reference-dependent; realized ROI is noisy; exchange liquidity buckets are incomplete. Future strategy changes require out-of-time validation.

**Workflow:** first verify Notebook 01 data readiness for the same snapshot, then execute this notebook. Subsequent visualizations and expanded segment analyses can build on these canonical tables.


In [ ]:
from __future__ import annotations

import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from oddsjam_ev.analysis.exchange_eval import build_exchange_capacity_table
from oddsjam_ev.analysis.filter_eval import FilterEvalConfig, build_filter_evaluation_table
from oddsjam_ev.analysis.scorecards import ExchangeScorecardConfig, build_exchange_scorecard

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


In [ ]:
AS_OF_DATE = "03-21-2026"
ARTIFACT_DIR = Path("../data/processed/artifacts")
WRITE_ARTIFACTS = False
processed_path = Path("../data/processed") / f"oddsjam-bet-tracker-processed-{AS_OF_DATE}.parquet"
if not processed_path.exists():
    raise FileNotFoundError(f"Processed snapshot not found: {processed_path}")
df_filtered = pd.read_parquet(processed_path)
print(f"Loaded {len(df_filtered):,} processed rows from {processed_path}")


## Exploratory research — Canonical filter evaluation tables
These reusable, regime-aware evaluation tables support the Notebook 02 research workflow. Keep execution-level versus opportunity-level semantics explicit, and do not treat production tags as evidence of strategy validity. These outputs are retained for historical comparability.

In [ ]:
# ------------------------------------------------------------------------------
# Canonical filter evaluation tables
# ------------------------------------------------------------------------------

eval_cfg = FilterEvalConfig()

filter_eval_all = build_filter_evaluation_table(
    df_filtered,
    cfg=eval_cfg,
    regime=None,
)

filter_eval_exchange = build_filter_evaluation_table(
    df_filtered,
    cfg=eval_cfg,
    regime="exchange",
)

filter_eval_sportsbook = build_filter_evaluation_table(
    df_filtered,
    cfg=eval_cfg,
    regime="sportsbook",
)

print("All filters:")
display(filter_eval_all.head(20))

print("Exchange filters:")
display(filter_eval_exchange.head(20))

print("Sportsbook filters:")
display(filter_eval_sportsbook.head(20))

In [ ]:
# ------------------------------------------------------------------------------
# Ranked views
# ------------------------------------------------------------------------------

rank_cols = [
    "saved_filter_names",
    "regime",
    "is_prod_filter",
    "bet_count",
    "total_stake",
    "total_profit",
    "total_actual_roi",
    "total_ev",
    "total_ev_roi",
    "profit_minus_ev",
    "ev_realization_gap",
    "edge_capture_ratio",
    "capture_band",
    "avg_clv",
    "max_drawdown",
    "roi_ci_low",
    "roi_ci_high",
    "duplicate_ratio",
    "sample_tier",
]

rank_cols = [c for c in rank_cols if c in filter_eval_all.columns]

exchange_ranked = (
    filter_eval_exchange[rank_cols]
    .sort_values(
        ["is_prod_filter", "total_actual_roi", "total_profit"],
        ascending=[False, False, False],
    )
    .reset_index(drop=True)
)

sportsbook_ranked = (
    filter_eval_sportsbook[rank_cols]
    .sort_values(
        ["is_prod_filter", "total_actual_roi", "total_profit"],
        ascending=[False, False, False],
    )
    .reset_index(drop=True)
)

print("Ranked exchange filters")
display(exchange_ranked.head(25))

print("Ranked sportsbook filters")
display(sportsbook_ranked.head(25))

## Exploratory research — Exchange capacity diagnostics
This section uses liquidity buckets as imperfect proxies for executable size. Sparse bucket coverage, quoted versus filled amounts, and filter overlap limit inference. The analysis is retained here for reproducibility.

In [ ]:
# ------------------------------------------------------------------------------
# Exchange capacity diagnostics
# ------------------------------------------------------------------------------

exchange_capacity = build_exchange_capacity_table(df_filtered)

display(exchange_capacity.head(25))

## Exploratory research — Exchange performance and capacity scorecard
Combines performance, production tags, duplicate diagnostics, and liquidity proxies. This is an exploratory prioritization view, **not** an automatic promotion or deployment rule. This analysis belongs in the filter-evaluation workflow (Notebook 02).

In [ ]:
# ------------------------------------------------------------------------------
# Exchange scorecard
# ------------------------------------------------------------------------------

scorecard_cfg = ExchangeScorecardConfig()

exchange_scorecard = build_exchange_scorecard(
    filter_eval_exchange=filter_eval_exchange,
    exchange_capacity=exchange_capacity,
    cfg=scorecard_cfg,
)

scorecard_cols = [
    "saved_filter_names",
    "regime",
    "is_prod_filter",
    "bet_count",
    "total_stake",
    "total_profit",
    "total_actual_roi",
    "total_ev",
    "total_ev_roi",
    "profit_minus_ev",
    "ev_realization_gap",
    "edge_capture_ratio",
    "capture_band",
    "avg_clv",
    "max_drawdown",
    "roi_ci_low",
    "roi_ci_high",
    "duplicate_ratio",
    "sample_tier",
    "liquidity_coverage",
    "avg_liquidity",
    "median_liquidity",
    "capacity_estimate",
    "capacity_ratio",
    "liquidity_weighted_ev",
    "capacity_to_stake_ratio",
]

scorecard_cols = [c for c in scorecard_cols if c in exchange_scorecard.columns]

exchange_scorecard = exchange_scorecard[scorecard_cols].reset_index(drop=True)

display(exchange_scorecard.head(25))

## Exploratory research — Exchange EV versus realized ROI plot
Expected and realized ROI are different estimands. Differences may reflect sampling variance, reference-price calibration, selection, or execution; they cannot be attributed to execution quality alone. Preserve this plot for Notebook 02.

In [ ]:
# ------------------------------------------------------------------------------
# Exchange filters: EV ROI vs realized ROI
# ------------------------------------------------------------------------------

plot_df = filter_eval_exchange.copy()

plot_df = plot_df[
    plot_df["total_ev_roi"].notna()
    & plot_df["total_actual_roi"].notna()
    & plot_df["bet_count"].notna()
].copy()

plt.figure(figsize=(10, 7))

prod_mask = (
    plot_df["is_prod_filter"].fillna(False).astype(bool)
    if "is_prod_filter" in plot_df.columns
    else pd.Series(False, index=plot_df.index)
)

# Non-prod first
plt.scatter(
    plot_df.loc[~prod_mask, "total_ev_roi"],
    plot_df.loc[~prod_mask, "total_actual_roi"],
    s=np.clip(plot_df.loc[~prod_mask, "bet_count"].fillna(0) * 2, 20, 400),
    alpha=0.55,
    label="test filters",
)

# Prod on top
plt.scatter(
    plot_df.loc[prod_mask, "total_ev_roi"],
    plot_df.loc[prod_mask, "total_actual_roi"],
    s=np.clip(plot_df.loc[prod_mask, "bet_count"].fillna(0) * 2, 20, 400),
    alpha=0.9,
    marker="D",
    label="prod filters",
)

plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)

for _, row in plot_df.iterrows():
    if row["bet_count"] >= 100:
        plt.annotate(
            row["saved_filter_names"],
            (row["total_ev_roi"], row["total_actual_roi"]),
            fontsize=8,
            alpha=0.8,
        )

plt.xlabel("Total EV ROI")
plt.ylabel("Total Realized ROI")
plt.title("Exchange Filters: EV ROI vs Realized ROI")
plt.legend()
plt.show()

## Exploratory research — Filter-level duplicate-ratio diagnostics
These diagnostics describe repeated signatures and possible opportunity overlap, not confirmed duplicate ingestion. Basic duplicate integrity is a Notebook 01 concern; performance-adjusted duplicate analysis is Notebook 02 research.

In [ ]:
# ------------------------------------------------------------------------------
# Duplicate ratio diagnostics
# ------------------------------------------------------------------------------

dup_diag_cols = [
    "saved_filter_names",
    "regime",
    "is_prod_filter",
    "bet_count",
    "n_unique_opportunities",
    "duplicate_ratio",
    "total_actual_roi",
    "total_ev_roi",
    "edge_capture_ratio",
    "capture_band",
]

dup_diag_cols = [c for c in dup_diag_cols if c in filter_eval_all.columns]

dup_diag = filter_eval_all[dup_diag_cols].copy()
dup_diag = dup_diag.sort_values("duplicate_ratio", ascending=False).reset_index(drop=True)

display(dup_diag.head(25))

In [ ]:
# ------------------------------------------------------------------------------
# Optional artifact writes
# ------------------------------------------------------------------------------

if WRITE_ARTIFACTS:
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    filter_eval_all.to_parquet(
        ARTIFACT_DIR / f"filter-eval-all-{AS_OF_DATE}.parquet",
        index=False,
    )
    filter_eval_exchange.to_parquet(
        ARTIFACT_DIR / f"filter-eval-exchange-{AS_OF_DATE}.parquet",
        index=False,
    )
    filter_eval_sportsbook.to_parquet(
        ARTIFACT_DIR / f"filter-eval-sportsbook-{AS_OF_DATE}.parquet",
        index=False,
    )
    exchange_capacity.to_parquet(
        ARTIFACT_DIR / f"exchange-capacity-{AS_OF_DATE}.parquet",
        index=False,
    )
    exchange_scorecard.to_parquet(
        ARTIFACT_DIR / f"exchange-scorecard-{AS_OF_DATE}.parquet",
        index=False,
    )

    print("Wrote regime-aware, prod-aware evaluation artifacts.")
else:
    print("WRITE_ARTIFACTS=False; no parquet artifacts written.")